# Women Misinformation Detector (Hackathon Notebook)

This notebook implements a **backend pipeline** for detecting misinformation about women using:
- **Mistral** (routing/classification) via raw HTTPS (no SDK dependencies)
- **Gemini** for claim extraction + **Google Search grounding** for citations

✅ No API keys are stored in this notebook. Add them at the top via environment variables.


In [ ]:
# (Optional) Install dependencies (run once)
# If pip on Windows is locked, restart kernel and rerun.
!pip -q install -U requests google-genai pandas weave wandb

In [ ]:
import os, re, json, time, hashlib
from pathlib import Path
import weave

# Initialize Weave tracing — all decorated functions will log to your W&B account
# Project: "femmestral-text-pipeline" so you can distinguish from finetune runs
weave.init(os.getenv("WANDB_PROJECT", "femmestral-text-pipeline"))

# ====== Models ======
MISTRAL_MODEL = os.getenv("MISTRAL_MODEL", "mistral-small-latest")
GEMINI_MODEL  = os.getenv("GEMINI_MODEL",  "gemini-2.5-flash")

# ====== Paths ======
INPUT_DIR  = Path("inputs")
OUTPUT_DIR = Path("outputs")
CACHE_DIR  = Path("cache")

for d in [INPUT_DIR, OUTPUT_DIR, CACHE_DIR]:
    d.mkdir(exist_ok=True)

print("Setup complete — Weave tracing enabled")
print("View traces at: https://wandb.ai/your-username/femmestral-text-pipeline/weave")

## Gemini grounding + strict JSON (fix for `Tool use with response_mime_type='application/json'`)

You hit this SDK limitation:

- When **tools** (Google Search grounding) are enabled, Gemini **cannot** return `response_mime_type="application/json"` in the **same** call.
- So we do **two steps**:
  1) **Grounded search (tools ON)** → returns normal **text**
  2) **Format to STRICT JSON (tools OFF)** → returns **JSON**

The next cells implement this with **debug prints** so you can see *inputs* and *outputs*.


In [35]:
# Cell G1: Imports + Gemini client setup (one-time)
# What this does:
# - Imports the Gemini SDK pieces we need for grounding
# - Creates `gclient` once so all later cells can use it

import os
import json
import re

from google import genai
from google.genai.types import GenerateContentConfig, Tool, GoogleSearch

# ✅ IMPORTANT: Set your key in the environment (DO NOT hardcode in shared notebooks)
# Example (temporary): os.environ["GEMINI_API_KEY"] = "YOUR_KEY"

assert "GEMINI_API_KEY" in os.environ and os.environ["GEMINI_API_KEY"].strip(),     "Missing GEMINI_API_KEY env var. Set it with: os.environ['GEMINI_API_KEY'] = '...'."

# Create the global Gemini client used everywhere
gclient = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Keep GEMINI_MODEL consistent:
# - If GEMINI_MODEL was defined earlier in the notebook, reuse it
# - Otherwise, default to env var or a sensible default
try:
    GEMINI_MODEL
except NameError:
    GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")

print("✅ Gemini client ready")
print("Model:", GEMINI_MODEL)

✅ Gemini client ready
Model: gemini-2.5-flash


In [36]:
# Cell G2: Why the error happened (simple demo)
# You previously used:
#   tools=[Tool(google_search=GoogleSearch())]
#   response_mime_type="application/json"
# in the same call, which triggers:
#   400 INVALID_ARGUMENT: Tool use with a response mime type 'application/json' is unsupported

print("✅ Fix = two-step pipeline")
print("Step A: tools ON (GoogleSearch) -> grounded TEXT")
print("Step B: tools OFF -> convert grounded TEXT into STRICT JSON")

✅ Fix = two-step pipeline
Step A: tools ON (GoogleSearch) -> grounded TEXT
Step B: tools OFF -> convert grounded TEXT into STRICT JSON


In [37]:
# Cell G3: Verification output schema (matches the rest of this notebook)
# This is the JSON shape we want AFTER grounding.
# NOTE: This schema is used in Step B (tools OFF), where JSON mime type IS allowed.

VERIFY_SCHEMA = {
  "type": "object",
  "properties": {
    "stance": {"type": "string", "enum": ["supported", "contradicted", "unclear"]},
    "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
    "rationale": {"type": "string"},
    "citations": {"type": "array", "items": {"type": "string"}}
  },
  "required": ["stance", "confidence", "rationale", "citations"]
}

print("✅ VERIFY_SCHEMA loaded")
print(json.dumps(VERIFY_SCHEMA, indent=2)[:400] + "...")

✅ VERIFY_SCHEMA loaded
{
  "type": "object",
  "properties": {
    "stance": {
      "type": "string",
      "enum": [
        "supported",
        "contradicted",
        "unclear"
      ]
    },
    "confidence": {
      "type": "string",
      "enum": [
        "low",
        "medium",
        "high"
      ]
    },
    "rationale": {
      "type": "string"
    },
    "citations": {
      "type": "array",
      "items...


In [38]:
# Cell G4: Helper to safely parse JSON even if the model adds extra text
# (Usually Step B returns clean JSON, but this avoids random demo failures.)

def _extract_json_from_text(s: str) -> dict:
    s = (s or "").strip()

    # Case 1: already pure JSON
    if s.startswith("{") and s.endswith("}"):
        return json.loads(s)

    # Case 2: find first JSON object inside text
    m = re.search(r"\{.*\}", s, re.DOTALL)
    if not m:
        raise ValueError("No JSON object found in model output.")
    return json.loads(m.group(0))

print("✅ JSON extraction helper ready")

✅ JSON extraction helper ready


In [ ]:
# Cell G5: Grounded claim verification (TWO STEPS) with DEBUG PRINTS
# Step A (tools ON): grounded text
# Step B (tools OFF): convert to STRICT JSON per VERIFY_SCHEMA

@weave.op()
def text_pipeline_verify_claim(claim_text: str, debug: bool = True) -> dict:
    # -----------------------
    # Step A: Grounded search
    # -----------------------
    grounded_prompt = f"""Verify the claim below using Google Search grounding.

Claim:
{claim_text}

Write a concise, evidence-based assessment.
Include:
- A 1-line stance suggestion: supported / contradicted / unclear
- 3-8 bullet points of key evidence (include nuance if needed)
- Source titles + URLs as plain text (so we can extract citations)
Prefer high-quality medical sources when relevant (WHO, CDC, NIH, NCBI, NHS, ACOG, Cochrane).
""".strip()

    if debug:
        print("\n" + "="*90)
        print("STEP A INPUT (grounded_prompt):")
        print(grounded_prompt[:1200] + ("..." if len(grounded_prompt) > 1200 else ""))
        print("="*90)

    grounded_resp = gclient.models.generate_content(
        model=GEMINI_MODEL,
        contents=grounded_prompt,
        config=GenerateContentConfig(
            temperature=0.0,
            tools=[Tool(google_search=GoogleSearch())],
        ),
    )

    grounded_text = (grounded_resp.text or "").strip()

    if debug:
        print("STEP A OUTPUT (grounded_text):")
        print(grounded_text[:1600] + ("..." if len(grounded_text) > 1600 else ""))
        print("="*90)

    # -----------------------
    # Step B: Format as JSON
    # -----------------------
    json_prompt = f"""Convert the grounded evidence into STRICT JSON that matches this schema:
{json.dumps(VERIFY_SCHEMA, indent=2)}

Rules:
- Output ONLY valid JSON. No markdown. No commentary.
- stance must be: supported, contradicted, or unclear
- confidence must be: low, medium, or high
- citations must be a list of URLs (strings), 2-6 items when possible

Grounded evidence:
{grounded_text}
""".strip()

    if debug:
        print("STEP B INPUT (json_prompt):")
        print(json_prompt[:1400] + ("..." if len(json_prompt) > 1400 else ""))
        print("="*90)

    json_resp = gclient.models.generate_content(
        model=GEMINI_MODEL,
        contents=json_prompt,
        config=GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",
            response_json_schema=VERIFY_SCHEMA,
        ),
    )

    raw = (json_resp.text or "").strip()

    if debug:
        print("STEP B OUTPUT RAW (should be JSON):")
        print(raw[:1600] + ("..." if len(raw) > 1600 else ""))
        print("="*90)

    try:
        out = json.loads(raw)
    except Exception:
        out = _extract_json_from_text(raw)

    if debug:
        print("PARSED JSON OUTPUT (dict):")
        print(json.dumps(out, indent=2)[:1600] + ("..." if len(json.dumps(out)) > 1600 else ""))
        print("="*90)

    return out

# Keep the old name as alias so downstream cells still work
verify_claim_grounded = text_pipeline_verify_claim

print("text_pipeline_verify_claim() ready — traced in Weave")

In [40]:
# Cell G6: Quick sanity test (optional)
# Run this AFTER you have a claim string you want to test.

test_claim = "HPV vaccines cause infertility in women."
try:
    test_out = verify_claim_grounded(test_claim, debug=True)
    test_out
except Exception as e:
    print("❌ Test failed:", repr(e))


STEP A INPUT (grounded_prompt):
Verify the claim below using Google Search grounding.

Claim:
HPV vaccines cause infertility in women.

Write a concise, evidence-based assessment.
Include:
- A 1-line stance suggestion: supported / contradicted / unclear
- 3–8 bullet points of key evidence (include nuance if needed)
- Source titles + URLs as plain text (so we can extract citations)
Prefer high-quality medical sources when relevant (WHO, CDC, NIH, NCBI, NHS, ACOG, Cochrane).
STEP A OUTPUT (grounded_text):
**Stance suggestion:** Contradicted

**Evidence-based assessment:**

The claim that HPV vaccines cause infertility in women is contradicted by extensive scientific evidence from leading global health organizations and numerous studies.

*   The World Health Organization (WHO) has concluded that available data do not support an association between HPV vaccination and infertility or primary ovarian insufficiency (POI), a condition where ovaries stop functioning before age 40.
*   The Cen

## Stage 1 — Create (synthetic) WhatsApp forward
You can replace this with any file coming from your frontend later.


In [41]:
dubai_msg = """🚨 Forwarded as received 🚨

Doctors in Dubai are warning women:
If you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility.

Instead:
✅ Drink hot water + turmeric
❌ Avoid painkillers for 3 days

This is from a “hospital circular”. Share this with every woman you know 🙏
"""

msg_path = INPUT_DIR / "dubai_forward_001.txt"
msg_path.write_text(dubai_msg, encoding="utf-8")
print("✅ Wrote:", msg_path)
print(dubai_msg)


✅ Wrote: inputs\dubai_forward_001.txt
🚨 Forwarded as received 🚨

Doctors in Dubai are warning women:
If you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility.

Instead:
✅ Drink hot water + turmeric
❌ Avoid painkillers for 3 days

This is from a “hospital circular”. Share this with every woman you know 🙏



## Stage 2 — Load raw content

In [42]:
raw_text = msg_path.read_text(encoding="utf-8", errors="ignore")
print("Loaded chars:", len(raw_text))
print(raw_text)


Loaded chars: 317
🚨 Forwarded as received 🚨

Doctors in Dubai are warning women:
If you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility.

Instead:
✅ Drink hot water + turmeric
❌ Avoid painkillers for 3 days

This is from a “hospital circular”. Share this with every woman you know 🙏



## Stage 3 — Canonicalize text (stable for hashing/dedup)

In [43]:
def canonicalize_text(text: str) -> str:
    t = text.lower().strip()
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"forwarded as received|fwd as received|copied message", "", t)
    t = re.sub(r"[!]{2,}", "!", t)
    t = re.sub(r"[?]{2,}", "?", t)
    return t.strip()

canon_text = canonicalize_text(raw_text)
print("Canonical chars:", len(canon_text))
print(canon_text)


Canonical chars: 292
🚨  🚨 doctors in dubai are warning women: if you take painkillers like ibuprofen during your period, it can "block" blood flow and cause infertility. instead: ✅ drink hot water + turmeric ❌ avoid painkillers for 3 days this is from a “hospital circular”. share this with every woman you know 🙏


## Stage 4 — Hashing (dedupe key)
We hash both raw and canonical text. Canonical hash is used to detect reprocessing of the *same* message with different formatting.


In [44]:
def sha256_hex(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

canonical_sha256 = sha256_hex(canon_text)
raw_sha256 = sha256_hex(raw_text)

print("canonical_sha256:", canonical_sha256)
print("raw_sha256      :", raw_sha256)


canonical_sha256: 9eefe6c6d7df5d542023e45f94a45d84b4b8aac85a0544e8c827b33baa2b79d0
raw_sha256      : 7369a3bb1e21f11e022359f34be70f42e7b67fc21abb9ac114e249ee42c2bc91


## Stage 4.5 — Cache check (avoid reprocessing)
If we already produced a Stage 8 verdict for this canonical hash, we can skip.


In [45]:
stage8_path = OUTPUT_DIR / f"{canonical_sha256}_stage8_verdict.json"
if stage8_path.exists():
    print("⚠️ Already processed. Found:", stage8_path)
    existing = json.loads(stage8_path.read_text(encoding="utf-8"))
    print("Final verdict:", existing.get("final"))
else:
    print("✅ Not processed before. Continue.")


⚠️ Already processed. Found: outputs\9eefe6c6d7df5d542023e45f94a45d84b4b8aac85a0544e8c827b33baa2b79d0_stage8_verdict.json
Final verdict: {'verdict': 'misinformation', 'confidence': 'high'}


## Stage 5 — Mistral routing/classification (no SDK; raw HTTPS)
This keeps the **Mistral theme** without pydantic dependency issues.


In [ ]:
import requests

ROUTER_SYSTEM = """You are a routing classifier for a misinformation detection pipeline focused on women.
Return ONLY valid JSON. No markdown.

Schema:
{
  "domain": "health|safety|legal|finance|bias|unknown",
  "risk_level": "low|medium|high",
  "route": "fast_check|deep_verify",
  "reasons": ["..."],
  "triggers": ["..."]
}

Rules:
- high risk if medical/legal/safety advice, infertility/cure claims, panic forwards, or instructions to share
- deep_verify if risk is medium or high, else fast_check
""".strip()

@weave.op()
def mistral_chat_json(system_prompt: str, user_text: str, model: str, max_tokens: int = 400) -> dict:
    api_key = os.getenv("MISTRAL_API_KEY")
    if not api_key:
        raise ValueError("Missing MISTRAL_API_KEY. Set it at the top of the notebook.")

    url = "https://api.mistral.ai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "temperature": 0.0,
        "max_tokens": max_tokens,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text}
        ],
    }
    r = requests.post(url, headers=headers, data=json.dumps(payload), timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"Mistral API error {r.status_code}: {r.text}")

    data = r.json()
    raw = data["choices"][0]["message"]["content"].strip()

    first, last = raw.find("{"), raw.rfind("}")
    if first == -1 or last == -1:
        raise ValueError("Mistral did not return JSON:\n" + raw)

    return json.loads(raw[first:last+1])

@weave.op()
def text_pipeline_route(text: str) -> dict:
    return mistral_chat_json(ROUTER_SYSTEM, text.strip(), model=MISTRAL_MODEL, max_tokens=400)

routing = text_pipeline_route(raw_text)
routing

## Stage 5 output — Save routing artifact

In [47]:
result_stage5 = {
    "pipeline_version": "v1",
    "trace_id": f"local-{int(time.time())}",
    "received_at": time.time(),
    "asset": {"path": str(msg_path), "type": "text"},
    "hashes": {"canonical_sha256": canonical_sha256, "raw_sha256": raw_sha256},
    "content": {"raw_preview": raw_text[:300], "canonical_text": canon_text},
    "routing_model": MISTRAL_MODEL,
    "routing": routing,
    "domain": routing.get("domain"),
    "risk_level": routing.get("risk_level"),
    "route": routing.get("route"),
}

stage5_path = OUTPUT_DIR / f"{canonical_sha256}_stage5.json"
stage5_path.write_text(json.dumps(result_stage5, indent=2), encoding="utf-8")
print("✅ Saved:", stage5_path)


✅ Saved: outputs\9eefe6c6d7df5d542023e45f94a45d84b4b8aac85a0544e8c827b33baa2b79d0_stage5.json


## Stage 6 — Claim extraction (Gemini structured JSON)
We extract atomic, checkable claims to verify individually.


In [48]:
import os
from google import genai
from google.genai.types import GenerateContentConfig, Tool, GoogleSearch

# Gemini client (expects GEMINI_API_KEY set in env)
if not os.getenv('GEMINI_API_KEY'):
    raise EnvironmentError(
        'Missing GEMINI_API_KEY. Set it as an environment variable before running.'
    )

gclient = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
print('✅ Gemini ready (gclient initialized)')


✅ Gemini ready (gclient initialized)


In [49]:
CLAIM_SCHEMA = {
  "type": "object",
  "properties": {
    "language": {"type": "string"},
    "claims": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "claim_id": {"type": "string"},
          "text": {"type": "string"},
          "claim_type": {
            "type": "string",
            "enum": ["medical_causal","medical_advice","authority_source","statistical","other"]
          },
          "check_priority": {"type": "string", "enum": ["low","medium","high"]},
          "entities": {"type": "array", "items": {"type": "string"}},
          "implied_advice": {"type": ["string", "null"]}
        },
        "required": ["claim_id","text","claim_type","check_priority","entities"]
      }
    }
  },
  "required": ["language","claims"]
}


In [ ]:
@weave.op()
def text_pipeline_extract_claims(text: str) -> dict:
    prompt = f"""You extract checkable claims from WhatsApp/Reddit messages related to women.

Return STRICT JSON that matches the provided schema.
Rules:
- Split into 2-6 atomic claims
- Include authority references (e.g., 'doctors said', 'hospital circular') as authority_source
- Include advice (do/don't) as medical_advice with implied_advice

Message:
{text}
""".strip()

    resp = gclient.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",
            response_json_schema=CLAIM_SCHEMA,
        ),
    )
    return json.loads(resp.text)

# Keep old name as alias
extract_claims_with_gemini = text_pipeline_extract_claims

claims_out = text_pipeline_extract_claims(raw_text)
claims_out

In [51]:
def validate_claims(obj: dict) -> None:
    assert isinstance(obj, dict), "Output is not a dict"
    assert "language" in obj and isinstance(obj["language"], str), "Missing/invalid language"
    assert "claims" in obj and isinstance(obj["claims"], list), "Missing/invalid claims list"
    for c in obj["claims"]:
        for k in ["claim_id","text","claim_type","check_priority","entities"]:
            assert k in c, f"Claim missing {k}"
        assert c["claim_type"] in ["medical_causal","medical_advice","authority_source","statistical","other"]
        assert c["check_priority"] in ["low","medium","high"]
        assert isinstance(c["entities"], list)

validate_claims(claims_out)
print("✅ Claims validated. Count =", len(claims_out["claims"]))


✅ Claims validated. Count = 6


In [52]:
for c in claims_out["claims"]:
    print(f"{c['claim_id']} | {c['claim_type']} | priority={c['check_priority']}")
    print("  ", c["text"])
    if c.get("implied_advice"):
        print("  advice:", c["implied_advice"])
    if c.get("entities"):
        print("  entities:", ", ".join(c["entities"]))
    print()


claim_1 | authority_source | priority=high
   Doctors in Dubai are warning women
  entities: Doctors in Dubai, women

claim_2 | medical_causal | priority=high
   taking painkillers like ibuprofen during your period can "block" blood flow
  entities: painkillers, ibuprofen, period, blood flow

claim_3 | medical_causal | priority=high
   taking painkillers like ibuprofen during your period can cause infertility
  entities: painkillers, ibuprofen, period, infertility

claim_4 | medical_advice | priority=medium
   Drink hot water + turmeric
  advice: It is advised to drink hot water + turmeric for period pain instead of painkillers.
  entities: hot water, turmeric

claim_5 | medical_advice | priority=high
   Avoid painkillers for 3 days
  advice: It is advised to avoid painkillers for 3 days during your period.
  entities: painkillers

claim_6 | authority_source | priority=high
   The information is from a "hospital circular"
  entities: hospital circular



In [53]:
stage6_path = OUTPUT_DIR / f"{canonical_sha256}_stage6_claims.json"
stage6_path.write_text(json.dumps(claims_out, indent=2), encoding="utf-8")
print("✅ Saved:", stage6_path)


✅ Saved: outputs\9eefe6c6d7df5d542023e45f94a45d84b4b8aac85a0544e8c827b33baa2b79d0_stage6_claims.json


## Stage 7–8 — Grounded verification (Gemini + Google Search)
This is the **judge-winning** step: each claim is checked with grounded web evidence and citations.


In [54]:
from google.genai.types import Tool, GoogleSearch

TRUSTED_DOMAINS = [
    "who.int", "cdc.gov", "nih.gov", "ncbi.nlm.nih.gov", "medlineplus.gov",
    "nhs.uk", "acog.org", "aafp.org", "cochrane.org",
    "mayoclinic.org", "clevelandclinic.org", "hopkinsmedicine.org"
]

from urllib.parse import urlparse

def domain_of(url: str) -> str:
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except Exception:
        return ""

def is_trusted(url: str) -> bool:
    d = domain_of(url)
    return any(d == td or d.endswith("." + td) for td in TRUSTED_DOMAINS)


In [55]:
# (Deprecated) Old one-shot tool+JSON approach removed
# Gemini SDK does NOT support using GoogleSearch tool AND response_mime_type="application/json"
# in the same call.
#
# Use verify_claim_grounded() defined earlier in cells G1–G5.

print("✅ Using the two-step verify_claim_grounded() defined above (tools ON -> text, tools OFF -> JSON).")

✅ Using the two-step verify_claim_grounded() defined above (tools ON -> text, tools OFF -> JSON).


In [56]:
per_claim_results = []
for c in claims_out["claims"]:
    v = verify_claim_grounded(c["text"])
    v["trusted_citations"] = [u for u in v.get("citations", []) if is_trusted(u)]
    per_claim_results.append({"claim": c, "verification": v})

per_claim_results[:1]



STEP A INPUT (grounded_prompt):
Verify the claim below using Google Search grounding.

Claim:
Doctors in Dubai are warning women

Write a concise, evidence-based assessment.
Include:
- A 1-line stance suggestion: supported / contradicted / unclear
- 3–8 bullet points of key evidence (include nuance if needed)
- Source titles + URLs as plain text (so we can extract citations)
Prefer high-quality medical sources when relevant (WHO, CDC, NIH, NCBI, NHS, ACOG, Cochrane).
STEP A OUTPUT (grounded_text):
**Stance Suggestion:** Supported

Doctors and health authorities in Dubai and the wider UAE are issuing various warnings and advice to women concerning health, safety, and legal matters.

*   Doctors in the UAE have emphasized the importance of regular health checks for women, especially those over 40, to screen for conditions like breast and cervical cancer, diabetes, high blood pressure, and high cholesterol.
*   There are warnings about the high prevalence of obesity, osteoporosis, and vi

[{'claim': {'claim_id': 'claim_1',
   'text': 'Doctors in Dubai are warning women',
   'claim_type': 'authority_source',
   'check_priority': 'high',
   'entities': ['Doctors in Dubai', 'women'],
   'implied_advice': None},
  'verification': {'stance': 'supported',
   'confidence': 'high',
   'rationale': "Doctors and health authorities in Dubai and the wider UAE are issuing various warnings and advice to women. This includes emphasizing regular health checks for women over 40 to screen for conditions like cancer, diabetes, and high blood pressure. They also warn about the prevalence of obesity, osteoporosis, and vitamin D deficiency, advising on diet and exercise. Dubai doctors have cautioned against using AI tools for self-diagnosis due to potential harm. Furthermore, authorities warn against unlicensed medical or cosmetic procedures due to safety risks, and the UAE's highest court has called for stricter regulations on cosmetic surgeries after a patient's death. Travel advisories al

In [ ]:
@weave.op()
def text_pipeline_aggregate(per_claim: list) -> dict:
    contradicted = sum(1 for x in per_claim if x["verification"]["stance"] == "contradicted")
    unclear = sum(1 for x in per_claim if x["verification"]["stance"] == "unclear")

    if contradicted >= 1:
        return {"verdict": "misinformation", "confidence": "medium" if contradicted == 1 else "high"}
    if unclear >= max(1, len(per_claim)//2):
        return {"verdict": "unclear", "confidence": "medium"}
    return {"verdict": "likely_true", "confidence": "medium"}

# Keep old name as alias
aggregate_verdict = text_pipeline_aggregate

final = text_pipeline_aggregate(per_claim_results)
final

In [58]:
def compose_shareback(final: dict, per_claim: list) -> str:
    trusted = []
    for x in per_claim:
        trusted += x["verification"].get("trusted_citations", [])
    trusted = list(dict.fromkeys(trusted))[:2]  # unique, first 2

    if final["verdict"] == "misinformation":
        header = "Quick fact-check: this forward looks misleading."
    elif final["verdict"] == "unclear":
        header = "Quick fact-check: this forward can’t be verified confidently."
    else:
        header = "Quick fact-check: this forward appears broadly consistent with reputable sources."

    bullets = [f"• {x['claim']['text']} → {x['verification']['stance']}" for x in per_claim[:3]]
    src_line = ("Sources: " + " | ".join(trusted)) if trusted else ""
    tail = "If this is health-related, consider checking with a clinician."

    parts = [header] + bullets + ([src_line] if src_line else []) + [tail]
    return "\n".join(parts)

shareback = compose_shareback(final, per_claim_results)
print(shareback)


Quick fact-check: this forward looks misleading.
• Doctors in Dubai are warning women → supported
• taking painkillers like ibuprofen during your period can "block" blood flow → contradicted
• taking painkillers like ibuprofen during your period can cause infertility → contradicted
Sources: https://collections.nlm.nih.gov/catalog/nlm:nlmuid-101600040-bk
If this is health-related, consider checking with a clinician.


In [59]:
stage8 = {
    "hashes": {"canonical_sha256": canonical_sha256, "raw_sha256": raw_sha256},
    "asset": {"path": str(msg_path), "type": "text"},
    "routing_model": MISTRAL_MODEL,
    "routing": routing,
    "claims": claims_out,
    "per_claim_verification": per_claim_results,
    "final": final,
    "shareback": shareback,
}

stage8_path = OUTPUT_DIR / f"{canonical_sha256}_stage8_verdict.json"
stage8_path.write_text(json.dumps(stage8, indent=2), encoding="utf-8")
print("✅ Saved:", stage8_path)


✅ Saved: outputs\9eefe6c6d7df5d542023e45f94a45d84b4b8aac85a0544e8c827b33baa2b79d0_stage8_verdict.json


In [60]:
# ===============================
# Cell R1: Realistic Reddit / Chat Messages
# ===============================

reddit_messages = [

"""Saw a post saying electric vehicles are actually worse for the environment
because battery production creates more pollution than gas cars ever will.
Feels like governments are hiding this.""",

"""My friend works in finance and told me Bitcoin transactions are totally
untraceable and governments can never track them.""",

"""Apparently NASA confirmed humans can see the Great Wall of China
from space without equipment. Crazy if true.""",

"""Someone told me WiFi routers consume massive electricity even when idle
and are one of the biggest causes of high electricity bills.""",

"""Read somewhere that AI will replace 90% of jobs within the next 5 years.
Seems inevitable honestly."""
]

print("✅ Loaded realistic Reddit-style messages")
print("Total messages:", len(reddit_messages))

✅ Loaded realistic Reddit-style messages
Total messages: 5


In [61]:
# ===============================
# Cell R2: Extract Claims
# ===============================

reddit_claim_sets = []

for msg in reddit_messages:

    print("\n====================================")
    print("REDDIT MESSAGE:")
    print(msg)

    claims = extract_claims_with_gemini(msg)

    print("\nEXTRACTED CLAIMS:")
    print(json.dumps(claims, indent=2))

    reddit_claim_sets.append(claims)


REDDIT MESSAGE:
Saw a post saying electric vehicles are actually worse for the environment
because battery production creates more pollution than gas cars ever will.
Feels like governments are hiding this.

EXTRACTED CLAIMS:
{
  "language": "en",
  "claims": [
    {
      "claim_id": "claim_1",
      "text": "electric vehicles are actually worse for the environment",
      "claim_type": "other",
      "check_priority": "high",
      "entities": [
        "electric vehicles",
        "environment"
      ],
      "implied_advice": null
    },
    {
      "claim_id": "claim_2",
      "text": "battery production creates more pollution than gas cars ever will",
      "claim_type": "other",
      "check_priority": "high",
      "entities": [
        "battery production",
        "pollution",
        "gas cars"
      ],
      "implied_advice": null
    },
    {
      "claim_id": "claim_3",
      "text": "governments are hiding this",
      "claim_type": "other",
      "check_priority": "high

In [62]:
# ===============================
# Cell R3: Verify Claims
# ===============================

reddit_results = []

for claim_block in reddit_claim_sets:
    for c in claim_block["claims"]:

        print("\n------------------------------------")
        print("VERIFYING CLAIM:")
        print(c["text"])

        verification = verify_claim_grounded(c["text"])

        print("\nVERIFICATION OUTPUT:")
        print(json.dumps(verification, indent=2))

        reddit_results.append({
            "claim": c["text"],
            "verification": verification
        })


------------------------------------
VERIFYING CLAIM:
electric vehicles are actually worse for the environment

STEP A INPUT (grounded_prompt):
Verify the claim below using Google Search grounding.

Claim:
electric vehicles are actually worse for the environment

Write a concise, evidence-based assessment.
Include:
- A 1-line stance suggestion: supported / contradicted / unclear
- 3–8 bullet points of key evidence (include nuance if needed)
- Source titles + URLs as plain text (so we can extract citations)
Prefer high-quality medical sources when relevant (WHO, CDC, NIH, NCBI, NHS, ACOG, Cochrane).
STEP A OUTPUT (grounded_text):
**Stance Suggestion:** Contradicted

**Key Evidence:**

*   While the manufacturing of electric vehicles (EVs), particularly their batteries, is more energy-intensive and generates higher initial carbon emissions than gasoline cars, EVs produce zero tailpipe emissions during operation.
*   Over their entire lifecycle, including manufacturing, operation, and di

In [ ]:
# Summary of Reddit claim verification results
for r in reddit_results:
    print("\n====================================")
    print("CLAIM:", r["claim"])
    print("STANCE:", r["verification"].get("stance", "N/A"))
    print("CONFIDENCE:", r["verification"].get("confidence", "N/A"))

    print("\nTOP SOURCES:")
    for url in r["verification"].get("citations", [])[:3]:
        print(" -", url)

    print("RATIONALE:", r["verification"].get("rationale", "N/A")[:200])